In [ ]:
import torch

print("Pytorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q uv

!uv python install 3.10

!uv venv \
    /content/dhwani-fairseq310 \
    --python 3.10 \
    --seed \
    --clear

PY = "/content/dhwani-fairseq310/bin/python"

print("Created fresh environment:", PY)

In [ ]:
import subprocess

PY = "/content/dhwani-fairseq310/bin/python"

subprocess.run([
    PY, "-m", "pip", "install",
    "pip<24.1",
    "setuptools==69.5.1",
    "wheel"
], check=True)

subprocess.run([
    PY, "-m", "pip", "install",
    "--index-url", "https://download.pytorch.org/whl/cu117",
    "torch==1.13.1+cu117",
    "torchaudio==0.13.1+cu117"
], check=True)

print("PyTorch stack installed")

In [ ]:
import subprocess

PY = "/content/dhwani-fairseq310/bin/python"

packages = [
    "numpy==1.26.4",
    "Cython==0.29.36",
    "omegaconf==2.0.6",
    "hydra-core==1.0.7",
    "bitarray",
    "cffi",
    "regex",
    "sacrebleu",
    "tqdm",
    "soundfile",
    "editdistance",
]

subprocess.run(
    [PY, "-m", "pip", "install", *packages],
    check=True
)

print("Dependencies installed")

In [ ]:
import subprocess
import shutil
from pathlib import Path

PY = "/content/dhwani-fairseq310/bin/python"
FAIRSEQ_DIR = Path("/content/ai4bharat-fairseq")

if FAIRSEQ_DIR.exists():
    shutil.rmtree(FAIRSEQ_DIR)

subprocess.run([
    "git", "clone",
    "--depth", "1",
    "https://github.com/AI4Bharat/fairseq.git",
    str(FAIRSEQ_DIR)
], check=True)

subprocess.run([
    PY, "-m", "pip", "install",
    "-e", str(FAIRSEQ_DIR),
    "--no-build-isolation",
    "--no-deps"
], check=True)

print("AI4Bharat Fairseq installed")

In [ ]:
import subprocess

PY = "/content/dhwani-fairseq310/bin/python"

code = """
import sys
import torch
import torchaudio
import fairseq

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("Fairseq:", fairseq.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
"""

result = subprocess.run(
    [PY, "-c", code],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print("STDERR:")
    print(result.stderr)

In [ ]:
from pathlib import Path

MODEL_DIR = Path("/content/indicwav2vec_telugu")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://objectstore.e2enetworks.net/indic-superb/aaai_ckpts/models/te"

CKPT_PATH = MODEL_DIR / "te.pt"
DICT_PATH = MODEL_DIR / "dict.ltr.txt"

print("Model directory:", MODEL_DIR)

In [ ]:
!wget -q --show-progress "$BASE_URL/te.pt" -O "$CKPT_PATH"
!wget -q --show-progress "$BASE_URL/dict.ltr.txt" -O "$DICT_PATH"

In [ ]:
import os

print(f"Checkpoint size: {os.path.getsize(CKPT_PATH) / (1024 ** 3):.2f} GB")
print("Dict exists:", DICT_PATH.exists())

with open(DICT_PATH, encoding="utf-8") as f:
    dict_lines = f.read().splitlines()

print("Dictionary symbols:", len(dict_lines))
print(dict_lines[:15])

In [ ]:
%pip install -q \
    datasets \
    jiwer \
    soundfile \
    librosa \
    huggingface_hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicVoices",
    "telugu",
    split="valid",
    streaming=True
)

num_samples = dataset.info.splits["valid"].num_examples

print(dataset)
print("IndicVoices Telugu validation samples:", num_samples)

In [ ]:
from pathlib import Path

WORK_DIR = Path("/content/indicwav2vec_work")
AUDIO_DIR = WORK_DIR / "audio"

AUDIO_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = WORK_DIR / "manifest.csv"
PRED_FILE = WORK_DIR / "predictions.csv"
DECODE_SCRIPT = WORK_DIR / "decode_fairseq.py"

TARGET_SR = 16000

print("Work directory:", WORK_DIR)
print("Audio directory:", AUDIO_DIR)
print("Manifest:", MANIFEST_FILE)

In [ ]:
import torchaudio
import soundfile as sf

def load_waveform(sample_item):
    audio = sample_item["audio_filepath"].get_all_samples()

    waveform = audio.data
    sample_rate = audio.sample_rate

    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sample_rate != TARGET_SR:
        waveform = torchaudio.functional.resample(
            waveform,
            orig_freq=sample_rate,
            new_freq=TARGET_SR
        )

    return waveform.float()

def normalize_for_wer(text):
    return " ".join(str(text).strip().split())

In [ ]:
import pandas as pd

if MANIFEST_FILE.exists():
    manifest_df = pd.read_csv(MANIFEST_FILE)
    dumped_indices = set(manifest_df["index"].astype(int).tolist())
    manifest_rows = manifest_df.to_dict("records")
else:
    dumped_indices = set()
    manifest_rows = []

print("Already dumped:", len(dumped_indices))
print("Remaining:", num_samples - len(dumped_indices))

In [ ]:
save_every = 100
new_since_save = 0

for index, sample_item in enumerate(dataset):

    if index in dumped_indices:
        continue

    waveform = load_waveform(sample_item)

    wav_path = AUDIO_DIR / f"{index:05d}.wav"

    sf.write(
        str(wav_path),
        waveform.squeeze().cpu().numpy(),
        TARGET_SR
    )

    manifest_rows.append({
        "index": index,
        "wav_path": str(wav_path),
        "speaker_id": sample_item["speaker_id"],
        "duration": sample_item["duration"],
        "reference": normalize_for_wer(sample_item["normalized"]),
    })

    dumped_indices.add(index)
    new_since_save += 1

    if new_since_save >= save_every:
        pd.DataFrame(manifest_rows).sort_values("index").to_csv(
            MANIFEST_FILE,
            index=False
        )

        print(f"Dumped {len(manifest_rows)}/{num_samples}")

        new_since_save = 0

pd.DataFrame(manifest_rows).sort_values("index").to_csv(
    MANIFEST_FILE,
    index=False
)

print("Audio dump complete:", len(manifest_rows), "files")

In [ ]:
import pandas as pd

manifest_df = pd.read_csv(MANIFEST_FILE)

wav_count = len(list(AUDIO_DIR.glob("*.wav")))
audio_gb = sum(p.stat().st_size for p in AUDIO_DIR.glob("*.wav")) / (1024 ** 3)

print("Manifest rows:", len(manifest_df))
print("Wav files on disk:", wav_count)
print(f"Audio size: {audio_gb:.2f} GB")
print("Expected:", num_samples)

manifest_df.head()

In [ ]:
%%writefile /content/indicwav2vec_work/decode_fairseq.py
import argparse
import csv
import os
import sys
import time

import torch
import torch.nn.functional as F
import soundfile as sf

from fairseq import checkpoint_utils
from fairseq.data.data_utils import post_process


def build_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument("--ckpt", required=True)
    parser.add_argument("--dict-dir", required=True)
    parser.add_argument("--manifest", required=True)
    parser.add_argument("--out", required=True)
    parser.add_argument("--limit", type=int, default=0)
    parser.add_argument("--save-every", type=int, default=25)
    return parser


def load_model(ckpt, dict_dir):
    try:
        models, saved_cfg, task = checkpoint_utils.load_model_ensemble_and_task(
            [ckpt],
            arg_overrides={
                "data": dict_dir,
                "task": {"data": dict_dir},
            },
        )
    except FileNotFoundError as e:
        print("FAILED to load checkpoint.", flush=True)
        print("Missing file:", e, flush=True)
        print(
            "This is usually cfg.model.w2v_path pointing at the pretrained "
            "checkpoint on the original training machine. Re-run with "
            "--w2v-path pointing at a local copy of that pretrained model.",
            flush=True,
        )
        raise

    model = models[0]
    model.eval()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    normalize = getattr(saved_cfg.task, "normalize", False)

    return model, task.target_dictionary, device, normalize


def decode_one(model, target_dict, device, normalize, wav_path):
    wav, sample_rate = sf.read(wav_path, dtype="float32")

    if wav.ndim > 1:
        wav = wav.mean(axis=1)

    source = torch.from_numpy(wav).float()

    if normalize:
        with torch.no_grad():
            source = F.layer_norm(source, source.shape)

    source = source.unsqueeze(0).to(device)
    padding_mask = torch.zeros_like(source, dtype=torch.bool)

    with torch.no_grad():
        encoder_out = model(source=source, padding_mask=padding_mask)

    emissions = encoder_out["encoder_out"]

    if emissions.shape[1] == 1:
        emissions = emissions.transpose(0, 1)

    emissions = emissions.float().cpu()

    toks = emissions[0].argmax(dim=-1).unique_consecutive()

    specials = {target_dict.bos(), target_dict.pad(), target_dict.eos()}
    toks = torch.tensor(
        [t for t in toks.tolist() if t not in specials], dtype=torch.long
    )

    hypothesis = target_dict.string(toks, extra_symbols_to_ignore=specials)

    return post_process(hypothesis, "letter").strip()


def main():
    args = build_parser().parse_args()

    with open(args.manifest, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    done = {}

    if os.path.exists(args.out):
        with open(args.out, encoding="utf-8") as f:
            for row in csv.DictReader(f):
                done[int(row["index"])] = row["prediction"]

    print("Manifest rows:", len(rows), flush=True)
    print("Already decoded:", len(done), flush=True)

    model, target_dict, device, normalize = load_model(
        args.ckpt,
        args.dict_dir
    )

    print("Device:", device, flush=True)
    print("Normalize:", normalize, flush=True)
    print("Vocabulary:", len(target_dict), flush=True)
    print("bos/pad/eos:", target_dict.bos(), target_dict.pad(), target_dict.eos(), flush=True)

    pending = [r for r in rows if int(r["index"]) not in done]

    if args.limit:
        pending = pending[: args.limit]

    print("To decode:", len(pending), flush=True)

    results = dict(done)
    processed = 0
    start = time.time()

    for row in pending:
        index = int(row["index"])

        prediction = decode_one(
            model,
            target_dict,
            device,
            normalize,
            row["wav_path"]
        )

        results[index] = prediction
        processed += 1

        elapsed = time.time() - start
        rate = processed / elapsed if elapsed > 0 else 0.0

        print(
            f"{index} | {processed}/{len(pending)} | "
            f"{rate:.2f} utt/s | {prediction[:60]}",
            flush=True
        )

        if args.limit == 0 and processed % args.save_every == 0:
            write_out(args.out, results)
            print(f"Checkpoint saved: {len(results)}", flush=True)

    if args.limit == 0:
        write_out(args.out, results)
        print("Final save:", len(results), flush=True)


def write_out(path, results):
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["index", "prediction"])

        for index in sorted(results):
            writer.writerow([index, results[index]])


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
import subprocess

PY = "/content/dhwani-fairseq310/bin/python"

sanity = subprocess.run(
    [
        PY, str(DECODE_SCRIPT),
        "--ckpt", str(CKPT_PATH),
        "--dict-dir", str(MODEL_DIR),
        "--manifest", str(MANIFEST_FILE),
        "--out", str(PRED_FILE),
        "--limit", "1",
    ],
    capture_output=True,
    text=True
)

print(sanity.stdout)

if sanity.returncode != 0:
    print("STDERR:")
    print(sanity.stderr[-4000:])

In [ ]:
print("REFERENCE :", manifest_df.iloc[0]["reference"])

In [ ]:
import subprocess

PY = "/content/dhwani-fairseq310/bin/python"

process = subprocess.Popen(
    [
        PY, str(DECODE_SCRIPT),
        "--ckpt", str(CKPT_PATH),
        "--dict-dir", str(MODEL_DIR),
        "--manifest", str(MANIFEST_FILE),
        "--out", str(PRED_FILE),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

process.wait()

print("Return code:", process.returncode)

In [ ]:
import pandas as pd

pred_df = pd.read_csv(PRED_FILE)
manifest_df = pd.read_csv(MANIFEST_FILE)

merged_df = manifest_df.merge(pred_df, on="index", how="inner")

merged_df["reference"] = merged_df["reference"].fillna("").map(normalize_for_wer)
merged_df["prediction"] = merged_df["prediction"].fillna("").map(normalize_for_wer)

print("Decoded samples:", len(merged_df))
print("Missing:", num_samples - len(merged_df))

merged_df.head()

In [ ]:
for i in range(5):
    print("REFERENCE :", merged_df.iloc[i]["reference"])
    print("PREDICTION:", merged_df.iloc[i]["prediction"])
    print()

In [ ]:
from jiwer import process_words

per_sample = []

for row in merged_df.to_dict("records"):
    score = process_words(row["reference"], row["prediction"])

    per_sample.append({
        "index": row["index"],
        "speaker_id": row["speaker_id"],
        "duration": row["duration"],
        "reference": row["reference"],
        "prediction": row["prediction"],
        "wer": score.wer,
        "substitutions": score.substitutions,
        "deletions": score.deletions,
        "insertions": score.insertions,
    })

results_df = pd.DataFrame(per_sample)

print("Rows scored:", len(results_df))

results_df.head()

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

results_dir = Path(
    "/content/drive/MyDrive/DhwaniLab/results/indicwav2vec"
)

results_dir.mkdir(parents=True, exist_ok=True)

checkpoint_file = results_dir / "indicvoices_telugu_valid.csv"

results_df.to_csv(checkpoint_file, index=False)

print("Saved rows:", len(results_df))
print("Saved to:", checkpoint_file)

In [ ]:
final_df = pd.read_csv(checkpoint_file)

references = final_df["reference"].fillna("").tolist()
predictions = final_df["prediction"].fillna("").tolist()

final_result = process_words(
    references,
    predictions
)

reference_words = (
    final_result.hits
    + final_result.substitutions
    + final_result.deletions
)

print("Samples:", len(final_df))
print("Reference words:", reference_words)
print("Correct words:", final_result.hits)
print("Substitutions:", final_result.substitutions)
print("Deletions:", final_result.deletions)
print("Insertions:", final_result.insertions)
print("Corpus WER:", final_result.wer)
print("Corpus WER %:", final_result.wer * 100)

In [ ]:
summary = {
    "model": "ai4bharat/IndicWav2Vec te.pt fairseq",
    "decoding": "greedy ctc",
    "dataset": "ai4bharat/IndicVoices telugu valid",
    "samples": len(final_df),
    "reference_words": int(reference_words),
    "hits": int(final_result.hits),
    "substitutions": int(final_result.substitutions),
    "deletions": int(final_result.deletions),
    "insertions": int(final_result.insertions),
    "corpus_wer": float(final_result.wer),
    "corpus_wer_percent": float(final_result.wer * 100),
}

summary_file = results_dir / "indicwav2vec_summary.csv"

pd.DataFrame([summary]).to_csv(summary_file, index=False)

print("Summary saved to:", summary_file)
print(summary)